In [1]:
"""
This example is uses the official
huggingface transformers `hyperparameter_search` API.
"""
import os

import ray
from ray import tune
from ray.tune import CLIReporter
from ray.tune.examples.pbt_transformers.utils import (
    download_data,
    build_compute_metrics_fn,
)
from ray.tune.schedulers import PopulationBasedTraining
from transformers import (
    glue_tasks_num_labels,
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    GlueDataset,
    GlueDataTrainingArguments,
    TrainingArguments,
)


def tune_transformer(num_samples=8, gpus_per_trial=0, smoke_test=False):
    data_dir_name = "./data" if not smoke_test else "./test_data"
    data_dir = os.path.abspath(os.path.join(os.getcwd(), data_dir_name))
    if not os.path.exists(data_dir):
        os.mkdir(data_dir, 0o755)

    # Change these as needed.
    model_name = (
        "bert-base-uncased" if not smoke_test else "sshleifer/tiny-distilroberta-base"
    )
    task_name = "rte"

    task_data_dir = os.path.join(data_dir, task_name.upper())

    num_labels = glue_tasks_num_labels[task_name]

    config = AutoConfig.from_pretrained(
        model_name, num_labels=num_labels, finetuning_task=task_name
    )

    # Download and cache tokenizer, model, and features
    print("Downloading and caching Tokenizer")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Triggers tokenizer download to cache
    print("Downloading and caching pre-trained model")
    AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
    )

    def get_model():
        return AutoModelForSequenceClassification.from_pretrained(
            model_name,
            config=config,
        )

    # Download data.
    download_data(task_name, data_dir)

    data_args = GlueDataTrainingArguments(task_name=task_name, data_dir=task_data_dir)

    train_dataset = GlueDataset(
        data_args, tokenizer=tokenizer, mode="train", cache_dir=task_data_dir
    )
    eval_dataset = GlueDataset(
        data_args, tokenizer=tokenizer, mode="dev", cache_dir=task_data_dir
    )

    training_args = TrainingArguments(
        output_dir=".",
        learning_rate=1e-5,  # config
        do_train=True,
        do_eval=True,
        no_cuda=gpus_per_trial <= 0,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        num_train_epochs=2,  # config
        max_steps=-1,
        per_device_train_batch_size=16,  # config
        per_device_eval_batch_size=16,  # config
        warmup_steps=0,
        weight_decay=0.1,  # config
        logging_dir="./logs",
        skip_memory_metrics=True,
        report_to="none",
    )

    trainer = Trainer(
        model_init=get_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=build_compute_metrics_fn(task_name),
    )

    tune_config = {
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 32,
        "num_train_epochs": tune.choice([2, 3, 4, 5]),
        "max_steps": 1 if smoke_test else -1,  # Used for smoke test.
    }

    scheduler = PopulationBasedTraining(
        time_attr="training_iteration",
        metric="eval_acc",
        mode="max",
        perturbation_interval=1,
        hyperparam_mutations={
            "weight_decay": tune.uniform(0.0, 0.3),
            "learning_rate": tune.uniform(1e-5, 5e-5),
            "per_device_train_batch_size": [16, 32, 64],
        },
    )

    reporter = CLIReporter(
        parameter_columns={
            "weight_decay": "w_decay",
            "learning_rate": "lr",
            "per_device_train_batch_size": "train_bs/gpu",
            "num_train_epochs": "num_epochs",
        },
        metric_columns=["eval_acc", "eval_loss", "epoch", "training_iteration"],
    )

    trainer.hyperparameter_search(
        hp_space=lambda _: tune_config,
        backend="ray",
        n_trials=num_samples,
        resources_per_trial={"cpu": 1, "gpu": gpus_per_trial},
        scheduler=scheduler,
#         keep_checkpoints_num=1,
#         checkpoint_score_attr="training_iteration",
        stop={"training_iteration": 1} if smoke_test else None,
        progress_reporter=reporter,
#         local_dir="~/ray_results/",
#         name="tune_transformer_pbt",
        log_to_file=False,
    )


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--smoke-test",
        default=True,
        action="store_true",
        help="Finish quickly for testing",
    )
    args, _ = parser.parse_known_args()

#     ray.init()

    if args.smoke_test:
        tune_transformer(num_samples=1, gpus_per_trial=0, smoke_test=True)
    else:
        # You can change the number of GPUs here:
        tune_transformer(num_samples=8, gpus_per_trial=1)

/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at sshleifer/tiny-distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Data already exists. Using downloaded data for task rte from /home/k2141255/research/study_design_classifier/study-design-classifier/test_data


/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/transformers/data/datasets/glue.py:88: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(
/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/transformers/data/processors/glue.py:520: FutureWarning: This processor will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(DEPRECATION_WARNING.format("processor"), FutureWarning)
/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/si

== Status ==
Current time: 2024-01-25 17:45:11 (running for 00:00:00.11)
PopulationBasedTraining: 0 checkpoints, 0 perturbs
Logical resource usage: 1.0/16 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2024-01-25_17-45-11
Number of trials: 1/1 (1 PENDING)
+------------------------+----------+-------+-----------+-------------+----------------+--------------+
| Trial name             | status   | loc   |   w_decay |          lr |   train_bs/gpu |   num_epochs |
|------------------------+----------+-------+-----------+-------------+----------------+--------------|
| _objective_7ce76_00000 | PENDING  |       |  0.238963 | 1.73374e-05 |             32 |            4 |
+------------------------+----------+-------+-----------+-------------+----------------+--------------+




(_objective pid=2782132) /home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
(_objective pid=2782132)   return self.fget.__get__(instance, owner)()
(_objective pid=2782132) Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at sshleifer/tiny-distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
(_objective pid=2782132) You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/9 [00:00<?, ?it/s]


Trial name,epoch,eval_acc,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,objective,should_checkpoint
_objective_7ce76_00000,0.01,0.472924,0.693154,0.0631,4388.67,142.592,0.472924,True


2024-01-25 17:45:13,722	INFO tune.py:1042 -- Total run time: 2.28 seconds (2.27 seconds for the tuning loop).


== Status ==
Current time: 2024-01-25 17:45:13 (running for 00:00:02.27)
PopulationBasedTraining: 0 checkpoints, 0 perturbs
Logical resource usage: 1.0/16 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: /home/k2141255/ray_results/_objective_2024-01-25_17-45-11
Number of trials: 1/1 (1 TERMINATED)
+------------------------+------------+-----------------------+-----------+-------------+----------------+--------------+------------+-------------+---------+----------------------+
| Trial name             | status     | loc                   |   w_decay |          lr |   train_bs/gpu |   num_epochs |   eval_acc |   eval_loss |   epoch |   training_iteration |
|------------------------+------------+-----------------------+-----------+-------------+----------------+--------------+------------+-------------+---------+----------------------|
| _objective_7ce76_00000 | TERMINATED | 10.151.122.12:2782132 |  0.238963 | 1.73374e-05 |             32 |            4 |   0.472924 |    0.69315

(_objective pid=2782132) /home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/transformers/data/metrics/__init__.py:61: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
(_objective pid=2782132)   warnings.warn(DEPRECATION_WARNING, FutureWarning)
(_objective pid=2782132) /home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/transformers/data/metrics/__init__.py:31: FutureWarning: This metric will be removed from the library soon, metrics should be handled with the 🤗 Evaluate library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
(_objective pid=2782132)   warnings.warn(DEPRECATIO